# Cyten backend benchmarks

Load JSON produced by `benchmark/run_benchmarks.py` and compare:

1. **Scaling** — wall time vs leg dimension (cyten backends/devices)
2. **Dense NumPy baseline** — `impl=numpy` rows from `--numpy-ref`
3. **Overhead** — ratio vs dense NumPy when present, else vs `nosym`+`numpy`+`cpu`
4. **Device speedup** — torch CUDA/MPS vs CPU

Requires: `matplotlib` (and optionally `pandas`).

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt

try:
    import pandas as pd
except ImportError:
    pd = None

# Change this to your results file (or a list of files to concatenate).
RESULT_PATHS = [
    Path('results/run.json'),
    # Path('results/smoke.json'),
]


def load_benchmark_file(path: Path):
    """Return (metadata, records) from a ``{metadata, results}`` JSON file."""
    raw = json.loads(path.read_text())
    return dict(raw['metadata'] or {}), list(raw['results'])


records = []
metadatas = []
for path in RESULT_PATHS:
    if not path.exists():
        print(f'missing: {path.resolve()}')
        continue
    meta, rows = load_benchmark_file(path)
    records.extend(rows)
    metadatas.append({'path': str(path), **meta})
    print(f'loaded {path} ({path.stat().st_size} bytes)')
    if meta:
        print(
            f"  cyten {meta.get('cyten_version')} commit={meta.get('cyten_commit_id')} "
            f"host={meta.get('hostname')} started={meta.get('started_at')}"
        )
        compile_info = meta.get('compile') or {}
        if compile_info.get('cmake_build_type'):
            print(
                f"  compile: {compile_info.get('cmake_build_type')} "
                f"{compile_info.get('cmake_cxx_flags_for_build_type')!r}"
            )

print(f'{len(records)} records')
cyten_records = [r for r in records if r.get('impl', 'cyten') == 'cyten']
numpy_records = [r for r in records if r.get('impl') == 'numpy']
print(f'  cyten={len(cyten_records)}  numpy={len(numpy_records)}')
if pd is not None and records:
    df = pd.DataFrame(records)
    display(df.head())
else:
    df = None


In [ ]:
def series_key(row):
    impl = row.get('impl', 'cyten')
    if impl == 'numpy':
        return f"numpy|{row['case']}"
    return (
        f"{row['case']}|{row['symmetry_backend']}|"
        f"{row['block_backend']}|{row['device']}"
    )


def group_by_op(recs):
    ops = sorted({r['op'] for r in recs})
    return {op: [r for r in recs if r['op'] == op] for op in ops}


def plot_scaling(recs, time_field='median'):
    by_op = group_by_op(recs)
    if not by_op:
        print('no data')
        return
    fig, axes = plt.subplots(1, len(by_op), figsize=(5.5 * len(by_op), 4.5), squeeze=False)
    for ax, (op, rows) in zip(axes[0], by_op.items()):
        series = {}
        for r in rows:
            series.setdefault(series_key(r), []).append(r)
        for name, pts in sorted(series.items()):
            pts = sorted(pts, key=lambda r: r['actual_dim'])
            xs = [r['actual_dim'] for r in pts]
            ys = [r[time_field] for r in pts]
            style = dict(marker='o', linestyle='--', color='0.35', linewidth=2) if name.startswith('numpy|') else dict(marker='o')
            ax.loglog(xs, ys, label=name, **style)
        ax.set_title(f'{op} scaling')
        ax.set_xlabel('actual leg dim')
        ax.set_ylabel(f'{time_field} time [s]')
        ax.grid(True, which='both', alpha=0.3)
        ax.legend(fontsize=7, loc='best')
    fig.tight_layout()
    plt.show()


plot_scaling(records)


## Overhead vs baseline

Prefer matching `impl=numpy` rows for the same `op` / `case` / `actual_dim`.
Fall back to cyten `nosym` + `numpy` + `cpu` when no dense reference is present.

In [ ]:
def find_baseline(recs, op, case, actual_dim, time_field='median'):
    numpy_cands = [
        r for r in recs
        if r.get('impl') == 'numpy'
        and r['op'] == op
        and r['case'] == case
    ]
    if numpy_cands:
        exact = [r for r in numpy_cands if r['actual_dim'] == actual_dim]
        if exact:
            return exact[0][time_field], 'numpy'
        nearest = min(numpy_cands, key=lambda r: abs(r['actual_dim'] - actual_dim))
        return nearest[time_field], 'numpy'

    cands = [
        r for r in recs
        if r.get('impl', 'cyten') == 'cyten'
        and r['op'] == op
        and r['case'] == 'nosym'
        and r['symmetry_backend'] == 'no_symmetry'
        and r['block_backend'] == 'numpy'
        and str(r['device']).startswith('cpu')
    ]
    if not cands:
        return None, None
    exact = [r for r in cands if r['actual_dim'] == actual_dim]
    if exact:
        return exact[0][time_field], 'nosym'
    nearest = min(cands, key=lambda r: abs(r['actual_dim'] - actual_dim))
    return nearest[time_field], 'nosym'


def plot_overhead(recs, time_field='median'):
    cyten = [r for r in recs if r.get('impl', 'cyten') == 'cyten']
    by_op = group_by_op(cyten)
    if not by_op:
        print('no cyten data')
        return
    fig, axes = plt.subplots(1, len(by_op), figsize=(5.5 * len(by_op), 4.5), squeeze=False)
    for ax, (op, rows) in zip(axes[0], by_op.items()):
        series = {}
        baseline_kind = None
        for r in rows:
            base, kind = find_baseline(recs, op, r['case'], r['actual_dim'], time_field=time_field)
            if base is None or base <= 0:
                continue
            baseline_kind = kind
            series.setdefault(series_key(r), []).append((r['actual_dim'], r[time_field] / base))
        for name, pts in sorted(series.items()):
            pts = sorted(pts)
            ax.semilogx([p[0] for p in pts], [p[1] for p in pts], marker='o', label=name)
        ax.axhline(1.0, color='k', ls='--', lw=1, alpha=0.5)
        kind = baseline_kind or 'baseline'
        ax.set_title(f'{op} overhead vs {kind}')
        ax.set_xlabel('actual leg dim')
        ax.set_ylabel(f'{time_field} time / baseline')
        ax.grid(True, which='both', alpha=0.3)
        ax.legend(fontsize=7, loc='best')
    fig.tight_layout()
    plt.show()


plot_overhead(records)


## Device speedup (torch)

For each `(op, case, symmetry_backend, dim)` with a torch CPU row, plot
`time_cpu / time_accel` for CUDA and MPS devices when present.

In [ ]:
def plot_device_speedup(recs, time_field='median'):
    torch_rows = [
        r for r in recs
        if r.get('impl', 'cyten') == 'cyten' and r['block_backend'] == 'torch'
    ]
    if not torch_rows:
        print('no torch rows')
        return

    def key(r):
        return (r['op'], r['case'], r['symmetry_backend'], r['actual_dim'])

    by_key = {}
    for r in torch_rows:
        by_key.setdefault(key(r), []).append(r)

    speedups = []
    for k, rows in by_key.items():
        cpu = [r for r in rows if str(r['device']).startswith('cpu')]
        if not cpu:
            continue
        t_cpu = cpu[0][time_field]
        for r in rows:
            dev = str(r['device'])
            if dev.startswith('cpu'):
                continue
            if r[time_field] <= 0:
                continue
            speedups.append({
                'op': k[0],
                'case': k[1],
                'symmetry_backend': k[2],
                'actual_dim': k[3],
                'device': dev,
                'speedup': t_cpu / r[time_field],
            })

    if not speedups:
        print('no CPU+accelerator pairs found (run with --devices cpu,cuda:0)')
        return

    ops = sorted({s['op'] for s in speedups})
    fig, axes = plt.subplots(1, len(ops), figsize=(5.5 * len(ops), 4.5), squeeze=False)
    for ax, op in zip(axes[0], ops):
        rows = [s for s in speedups if s['op'] == op]
        series = {}
        for s in rows:
            name = f"{s['case']}|{s['symmetry_backend']}|{s['device']}"
            series.setdefault(name, []).append((s['actual_dim'], s['speedup']))
        for name, pts in sorted(series.items()):
            pts = sorted(pts)
            ax.semilogx([p[0] for p in pts], [p[1] for p in pts], marker='o', label=name)
        ax.axhline(1.0, color='k', ls='--', lw=1, alpha=0.5)
        ax.set_title(f'{op} torch speedup (CPU time / device time)')
        ax.set_xlabel('actual leg dim')
        ax.set_ylabel('speedup')
        ax.grid(True, which='both', alpha=0.3)
        ax.legend(fontsize=7, loc='best')
    fig.tight_layout()
    plt.show()


plot_device_speedup(records)


## Quick table

Optional pandas summary at the largest `actual_dim` per series.

In [ ]:
if df is None:
    print('pandas not installed; skip table')
elif df.empty:
    print('no records')
else:
    cols = ['op', 'impl', 'case', 'symmetry_backend', 'block_backend', 'device']
    idx = df.groupby(cols)['actual_dim'].idxmax()
    summary = df.loc[idx, cols + [
        'dim', 'actual_dim', 'num_blocks', 'median', 'mean', 'stdev',
    ]].sort_values(cols)
    display(summary)
